# Preprocessing Data for NLP Tasks

## Setup

In [5]:
!python --version

Python 3.12.9


In [2]:
!cat requirements.txt

# Python 3.12.x
pandas==2.3.3
numpy==1.26.4
matplotlib==3.10.6
seaborn==0.13.2
wordcloud==1.9.4
nltk==3.9.2
spacy==3.8.9
textblob==0.19.0
scikit-learn==1.7.2
bertopic==0.17.3


In [6]:
!python -m pip install -q -r requirements.txt

In [7]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.5 MB/s eta 0:00:000:00:010:00:01:01m
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


### Imports

In [1]:
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from bertopic import BERTopic
import pandas as pd

### Configurations

In [3]:
data_path = '../data/cleaned_ntsb.csv'

In [4]:
pd.set_option('display.max_columns', None)

## Load Data

In [5]:
df = pd.read_csv(data_path, low_memory=False)
df.shape

(87951, 47)

In [6]:
df.head()

,event_id,event_date,investigation_type,country,aircraft_damage,aircraft_category,make,model,amateur_built,number_of_engines,engine_type,far_description,schedule,purpose_of_flight,total_fatal_injuries,total_serious_injuries,total_minor_injuries,total_uninjured,weather_condition,broad_phase_of_flight,analysis,analysis_clean,city,longitude,latitude,address,geometry,place,number_of_seats,type_aircraft,type_engine,total_person,far_description_factorized,schedule_factorized,purpose_of_flight_factorized,make_factorized,model_factorized,year,publication_year,month,publication_month,day,publication_day,date_difference,publication_month_name,event_month_name,season
0,20001218X45444,1948-10-24,Accident,United States,Destroyed,fixed wing single engine,stinson,108-3,No,1,reciprocating,part 91: general aviation,UNK,Personal,2,0,0,0,UNK,Cruise,"ON OCTOBER 24, 1948, THE PILOT DEPARTED A PRIV...",on october the pilot departed a private airstr...,moose creek,-147.160665,64.713512,"Moose Creek, Fairbanks North Star, Alaska, Uni...",POINT (-147.1606646266588 64.7135123),mountain,4,4,1,2,1,2,0,0,0,1948,2001,10,8,24,24.0,26,August,October,Fall
1,20001218X45447,1962-07-19,Accident,United States,Destroyed,weight-shift-control,piper,pa24-180,No,1,reciprocating,part 91: general aviation,UNK,Personal,4,0,0,0,UNK,Unknown,"ON JULY 19, 1962, A COMMERCIAL, NON-INSTRUMENT...",on july a commercial non instrument rated pilo...,bridgeport,-73.188786,41.179269,"Bridgeport, Greater Bridgeport Planning Region...",POINT (-73.1887863 41.1792695),sea,4,7,1,4,1,2,0,1,1,1962,1996,7,9,19,19.0,34,September,July,Summer
2,20061025X01555,1974-08-30,Accident,United States,Destroyed,fixed wing single engine,cessna,172m,No,1,reciprocating,part 91: general aviation,UNK,Personal,3,0,0,1,IMC,Cruise,The private pilot was issued his certificate o...,the private pilot was issued his certificate o...,saltville,-81.762063,36.881503,"Saltville, Smyth County, Virginia, United States",POINT (-81.7620635 36.8815031),sea,4,4,1,3,1,2,0,2,2,1974,2007,8,2,30,30.0,33,February,August,Summer
3,20001218X45448,1977-06-19,Accident,United States,Destroyed,weight-shift-control,rockwell,112,No,1,reciprocating,part 91: general aviation,UNK,Personal,2,0,0,0,IMC,Cruise,The aircraft wreckage was discovered 22 miles ...,the aircraft wreckage was discovered miles sou...,eureka,-124.167375,40.790687,"Eureka, Humboldt County, California, United St...",POINT (-124.1673746 40.7906871),airport,4,7,1,2,1,2,0,3,3,1977,2000,6,12,19,19.0,23,December,June,Summer
4,20041105X01764,1979-08-02,Accident,United States,Destroyed,fixed wing multi engine,cessna,501,No,2,turbo fan,part 91: general aviation,UNK,Personal,1,2,0,0,VMC,Approach,The Safety Board's full report is available at...,the safety board s full report is available at...,canton,-95.864051,32.555664,"Canton, Van Zandt County, Texas, 75103, United...",POINT (-95.8640507 32.555664),airport,8,5,5,3,1,2,0,2,4,1979,1980,8,4,2,2.0,1,April,August,Summer


## Sentiment Analysis

In [7]:
df["sentiment_polarity"] = df["analysis_clean"].apply(lambda x: TextBlob(x).sentiment.polarity)
df["sentiment_category"] = pd.cut(
    df["sentiment_polarity"],
    bins=[-1, -0.05, 0.05, 1],
    labels=["Negative", "Neutral", "Positive"]
)

In [8]:
df[["analysis_clean", "sentiment_polarity", "sentiment_category"]].sample(50, random_state=42)

,analysis_clean,sentiment_polarity,sentiment_category
25609,witnesses observed the acft maneuvering at a l...,-0.019136,Neutral
31738,factual,0.000000,Neutral
5346,after takeoff the plt of the ultralight vehicl...,0.020833,Neutral
85707,the pilot had made two flights on the day of t...,0.024351,Neutral
59785,the airplane was flown by the student and cert...,0.024451,Neutral
22491,the plt landed in a field which he knew contai...,0.214286,Positive
25703,the pilot was landing and bounced twice he rej...,0.158929,Positive
7492,the plt angled in on the final approach at low...,0.095238,Positive
64569,same as factual information,0.000000,Neutral
4293,the pilot was warming the eng for a compressio...,0.037143,Neutral


### Analyze Sentiment Spread

In [9]:
sentiment_counts = df["sentiment_category"].value_counts()
sentiment_ratio = sentiment_counts / len(df)
sentiment_percent = sentiment_ratio.map(lambda x: f'{x*100:.2f}%')
sentiment_spread = pd.DataFrame({
    'Count': sentiment_counts,
    'Ratio': sentiment_ratio,
    'Percentage': sentiment_percent,
})
sentiment_spread

,Count,Ratio,Percentage
sentiment_category,,,
Neutral,38948,0.442837,44.28%
Positive,27845,0.316597,31.66%
Negative,21158,0.240566,24.06%


**Observation:** Slight imbalance of sentiment categories across the dataset with `Neutral` sentiment representing the majority and `Negative` representing the minority. A perfect balance would have been approximately `33.33%` per category.

In [10]:
# Inspect one report from each category
for category in df["sentiment_category"].unique():
    item = df[df["sentiment_category"] == category].iloc[0]
    narrative = item['analysis_clean']
    event_id = item['event_id']
    event_date = item['event_date']
    print('='*60)
    print(f'Category   : {category}')
    print(f'Event ID   : {event_id}')
    print(f'Event Date : {event_date}')
    print(f'Narrative  : {narrative}')
print('='*60)

Category   : Neutral
Event ID   : 20001218X45444
Event Date : 1948-10-24
Narrative  : on october the pilot departed a private airstrip at the moose creek ranch at two other pilots that departed at the same time reported encountering heavy snow squalls along the selway river as they headed westbound the wreckage was found in april miles west of moose creek in the selway river drainage in mountainous terrain the wreckage exhibited characteristics consistent with impact with trees and terrain in cruise flight 
Category   : Negative
Event ID   : 20001218X45447
Event Date : 1962-07-19
Narrative  : on july a commercial non instrument rated pilot and three passengers were crossing high mountainous terrain on a night cross country flight subsequently the airplane collided with rising terrain at about feet mean sea level wreckage was found on a degree slope with extensive damage the airplane was reported missing and went undiscovered until august the airplane with human remains was discovered i

## Keyword Extraction

In [11]:
vectorizer = TfidfVectorizer(max_df=0.8, min_df=10, stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["analysis_clean"])
feature_names = vectorizer.get_feature_names_out()

In [12]:
# Get top words by average TF-IDF score
tfidf_mean = tfidf_matrix.mean(axis=0).A1
keywords = pd.DataFrame({"word": feature_names, "tfidf": tfidf_mean})
keywords.sort_values("tfidf", ascending=False).head(20)

,word,tfidf
333,airplane,0.068049
6847,pilot,0.059682
5243,landing,0.039301
3253,engine,0.038957
8128,runway,0.037438
3993,fuel,0.035504
3552,factual,0.032002
5329,left,0.030863
97,acft,0.030309
3804,flight,0.029862


## Topic Modeling

### BERT Topic

In [13]:
topic_model = BERTopic(language="english")
topics, probs = topic_model.fit_transform(df["analysis_clean"])

df["topic"] = topics
topic_info = topic_model.get_topic_info()
topic_info.head(10)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,26633,-1_airplane_runway_pilot_and,"[airplane, runway, pilot, and, the, that, engi...",[on the morning of the accident the student pi...
1,0,14320,0_acft_plt_rwy_flt,"[acft, plt, rwy, flt, eng, arpt, ft, aprx, apc...",[the plt stated that after takeoff from rwy he...
2,1,6370,1_helicopter_rotor_tail_collective,"[helicopter, rotor, tail, collective, autorota...",[the pilot reported that while in a hover abou...
3,2,3791,2_student_instructor_solo_cfi,"[student, instructor, solo, cfi, runway, go, a...",[the solo student pilot reported that during f...
4,3,3414,3_factual_same_information_as,"[factual, same, information, as, narrative, fo...","[same as factual information, same as factual ..."
5,4,2751,4_fuel_tank_gallons_tanks,"[fuel, tank, gallons, tanks, selector, engine,...",[the pilot reported that during preflight prep...
6,5,1939,5_foreign___,"[foreign, , , , , , , , , ]","[foreign, foreign, foreign]"
7,6,1611,6_knots_wind_winds_degrees,"[knots, wind, winds, degrees, gusting, runway,...",[the pilot reported that during final approach...
8,7,1196,7_oil_connecting_rod_crankshaft,"[oil, connecting, rod, crankshaft, engine, bea...",[while maneuvering during the instructional fl...
9,8,1112,8_unk___,"[unk, , , , , , , , , ]","[unk, unk, unk]"


### LDA with Visualization (Gensim)